# Tray lane allocation model for Q4

This notebook reads the `*.json` instance and builds the MILP model in Pyomo.


In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
import pyomo.environ as pyo


## Load data


In [ ]:
with open(Path("data/16_trays.json")) as f:
    data = json.load(f)


In [ ]:
tray_types = {
    t["tray_type_name"]: t
    for t in data["tray_types"]
}

trays = data["trays"]

tray_lengths = {
    tray["id"]: tray_types[tray["tray_type"]]["length"]
    for tray in trays
}


## Basic sets and modelling parameter

We use at most `n` candidate lanes, where `n` is the number of trays.  
This is a standard modelling assumption, since in the worst case each tray can be placed in its own lane.

`K` is the maximum number of lanes allowed in the current model.


In [ ]:
I = sorted(tray_lengths.keys())
J = list(range(len(I)))

total_tray_length = sum(tray_lengths.values())
K_start = max(1, math.floor(total_tray_length / 2))
target_Z = 2.0
n = len(I)


## Build the Pyomo model


In [ ]:
def build_model(K):
    model = pyo.ConcreteModel(name="tray_lane_allocation")

    model.I = pyo.Set(initialize=I)
    model.J = pyo.Set(initialize=J)

    model.l = pyo.Param(model.I, initialize=tray_lengths, within=pyo.NonNegativeReals)
    model.K = pyo.Param(initialize=K, within=pyo.NonNegativeIntegers)

    model.x = pyo.Var(model.I, model.J, domain=pyo.Binary)
    model.y = pyo.Var(model.J, domain=pyo.Binary)
    model.L = pyo.Var(model.J, domain=pyo.NonNegativeReals)
    model.Z = pyo.Var(domain=pyo.NonNegativeReals)

    model.assign_con = pyo.ConstraintList()
    for i in model.I:
        model.assign_con.add(sum(model.x[i, j] for j in model.J) == 1)

    model.length_con = pyo.ConstraintList()
    for j in model.J:
        model.length_con.add(model.L[j] == sum(model.l[i] * model.x[i, j] for i in model.I))

    model.link_con = pyo.ConstraintList()
    for i in model.I:
        for j in model.J:
            model.link_con.add(model.x[i, j] <= model.y[j])

    model.max_con = pyo.ConstraintList()
    for j in model.J:
        model.max_con.add(model.L[j] <= model.Z)

    model.lane_limit_con = pyo.ConstraintList()
    model.lane_limit_con.add(sum(model.y[j] for j in model.J) <= model.K)

    model.obj = pyo.Objective(expr=model.Z, sense=pyo.minimize)
    return model


## Solve for increasing lane limits


In [ ]:
tray_type_map = {
    tray["id"]: tray["tray_type"]
    for tray in trays
}

In [ ]:
solver = pyo.SolverFactory("cbc")
solver.options["seconds"] = 10
solver.options["ratio"] = 0.01

target_Z = 2.0
best_result = None
all_results = []

for K in range(K_start, n + 1):
    model = build_model(K)
    results = solver.solve(model, tee=True)

    termination = results.solver.termination_condition

    try:
        z_value = pyo.value(model.Z)
    except:
        print(f"K={K}, no feasible solution found.")
        all_results.append({
            "K": K,
            "Z": None,
            "termination": str(termination),
            "lane_summary": None,
        })
        continue

    # extract tray-to-lane solution
    rows = []
    for i in model.I:
        for j in model.J:
            if pyo.value(model.x[i, j]) > 0.5:
                rows.append({
                    "tray_id": int(i),
                    "lane_id": int(j),
                    "type": tray_type_map[int(i)],
                    "length": float(pyo.value(model.l[i])),
                })

    solution_df = pd.DataFrame(rows)

    lane_summary = (
        solution_df
        .groupby(["lane_id", "type"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["small", "medium", "large"]:
        if col not in lane_summary.columns:
            lane_summary[col] = 0

    lane_lengths = (
        solution_df
        .groupby("lane_id")["length"]
        .sum()
        .reset_index(name="lane_length")
    )

    lane_summary = (
        lane_summary
        .merge(lane_lengths, on="lane_id")
        [["lane_id", "small", "medium", "large", "lane_length"]]
        .sort_values("lane_id")
        .reset_index(drop=True)
    )

    print(f"K={K}, Z={z_value:.4f}, termination={termination}")
    print(lane_summary)

    result_entry = {
        "K": K,
        "Z": z_value,
        "termination": str(termination),
        "solution_df": solution_df,
        "lane_summary": lane_summary,
        "model": model,
        "results": results,
    }

    all_results.append(result_entry)

    best_result = result_entry

    if z_value <= target_Z:
        print(f"Stop: found solution with K={K} and Z={z_value:.4f} <= {target_Z}")
        break

tradeoff_df = pd.DataFrame([
    {
        "K": r["K"],
        "Z": r["Z"],
        "termination": r["termination"],
    }
    for r in all_results
])

tradeoff_df

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

export_paths = []

for idx, result in enumerate(all_results[:2], start=1):
    df = result["solution_df"][["tray_id", "lane_id"]].copy()

    lane_ids = sorted(df["lane_id"].unique())
    lane_map = {old: new for new, old in enumerate(lane_ids)}
    df["lane_id"] = df["lane_id"].map(lane_map)

    df = df.sort_values(["lane_id", "tray_id"])

    path = output_dir / f"option_{idx}.csv"
    df.to_csv(path, index=False)

    export_paths.append(path)

export_paths

In [ ]:
all_results[1]["lane_summary"]